# Setup Bronze tables

In [0]:
%reload_ext autoreload
%autoreload 2

import importlib
import etl_config.bronze_config as bronze_config_module

importlib.reload(bronze_config_module)

# Bronze table configuration

- Should be executed on setup
- Creating bronze table with base schema
- Compact existing small files
- Set table properties to enable auto compaction and optimize future writes
- Set columns description

In [0]:
from utils.logger import get_logger
from utils.ddl import create_bronze_schema
from etl_config.bronze_config import BRONZE_CONFIG

logger = get_logger("bronze_setup")

### Create bronze tables

In [0]:
for table_key, cfg in BRONZE_CONFIG.items():
    table_name = cfg.table_name

    if not spark.catalog.tableExists(table_name):
        logger.info(f"Creating bronze table: {table_name}")

        # Create schema from required columns
        schema = create_bronze_schema(cfg.required_columns)

        # Create table
        df = spark.createDataFrame([], schema)
        df.write.format("delta").saveAsTable(table_name)

        # Set table properties to enable auto compaction and optimize future writes
        spark.sql(
            f"""
                ALTER TABLE {table_name}
                SET TBLPROPERTIES (
                    'delta.autoOptimize.optimizeWrite' = 'true',
                    'delta.autoOptimize.autoCompact' = 'true',
                    'delta.columnMapping.mode' = 'name',
                    'description' = :table_description
                )
            """,
            args={"table_description": cfg.table_description}
        )

        logger.info(f"Created: {table_name}")

    else:
        logger.info(f"Table already exists: {table_name}")

    # Add comments
    for column, comment in cfg.column_comments.items():
        try:
            spark.sql(
                f"COMMENT ON COLUMN {table_name}.{column} IS :comment",
                args={"comment": comment}
            )
            logger.info(f"Added comment to column: {table_name}.{column}")
        except Exception as e:
            logger.info(f"Column still does not exist: {table_name}.{column}")
            continue


In [0]:
%sql
use acme_catalog.bronze;

select * from information_schema.tables where table_catalog = 'acme_catalog' and table_schema = 'bronze' order by table_name;

select * from information_schema.columns where table_name = 'products' order by column_name;